# Bibliography Extraction

Parse messy OCR/HTML bibliography into structured data using LLMs.

In [1]:
from largeliterarymodels.tasks import BibliographyTask, BibliographyEntry

task = BibliographyTask()
task

BibliographyTask(name='BibliographyTask', schema=list[BibliographyEntry])

In [3]:
# task.stash.clear()

## Load and slice the bibliography

Replace the path below with your OCR/HTML file. Then split into chunks — one chunk per year heading is a natural boundary.

In [4]:
import re

# Load your file
biblio_path = "/Users/ryan/Downloads/English prose fiction, 1600-1700 _ a chronological checklist.html"
with open(biblio_path) as f:
    raw_html = f.read()

# Split on year headings (e.g. <h2...><span...>1600</span></h2>)
# Each chunk keeps its heading so the LLM knows the year
chunks = re.split(r'(?=<h2[^>]*>)', raw_html)
chunks = [c.strip() for c in chunks if c.strip() and '<h2' in c]

print(f"{len(chunks)} chunks")
for c in chunks[:3]:
    print(f"  {c[:80]}...")
print('...')
for c in chunks[-3:]:
    print(f"  {c[:80]}...")

91 chunks
  <h2 style="font-weight:normal;"><a id="bookmark3"></a><span class="font5">1600</...
  <h2 style="font-weight:normal;"><a id="bookmark4"></a><span class="font5">1601</...
  <h2 style="font-weight:normal;"><a id="bookmark5"></a><span class="font5">1603</...
...
  <h2 style="font-weight:normal;"><a id="bookmark91"></a><span class="font5">1698<...
  <h2 style="font-weight:normal;"><a id="bookmark92"></a><span class="font5">^99</...
  <h2 style="font-weight:normal;"><a id="bookmark93"></a><span class="font5">I 700...


## Run extraction

Run a single chunk first to verify, then batch all chunks.

In [6]:
# Test on one chunk
entries = task.run(chunks[0])
for e in entries[:3]:
    print(f"  {e.year} | {e.author} | {e.title} | printer={e.printer} | pub={e.publisher}")


  1600 | Albions Queen | The famous history of Albions queene | printer=W. White | pub=T. Pavier
  1600 | Armin, Robert | Foole upon foole, or Six sortes of sottes | printer= | pub=W. Ferbrand
  1600 | Breton, Nicholas | The strange fortunes of two excellent princes | printer=P. Short | pub=N. Ling


In [ ]:
# Run all chunks (cached — safe to re-run)
all_entries = task.map(chunks)

# Flatten: each chunk returns a list of BibliographyEntry
flat = [entry for chunk_entries in all_entries for entry in chunk_entries]
print(f"{len(flat)} total entries extracted from {len(chunks)} chunks")

## Export to DataFrame / CSV

In [ ]:
import pandas as pd

df = pd.DataFrame([e.model_dump() for e in flat])
df = df.sort_values(["year", "author"]).reset_index(drop=True)
print(f"{len(df)} rows, {len(df.columns)} columns")
df.head(10)

In [ ]:
# Save to CSV
output_path = "../data/bibliography.csv"
df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

## Compare models (optional)

Run the same extraction with a different model to compare results.

In [ ]:
# Compare on a single chunk
for model in ["claude-sonnet-4-20250514", "gpt-4o-mini", "gemini-2.5-flash"]:
    entries = task.run(chunks[0], model=model)
    print(f"\n{model}: {len(entries)} entries")
    for e in entries[:2]:
        print(f"  {e.year} | {e.author} | {e.title}")